In [1]:
import os

import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras.models as M
import tensorflow.keras.backend as K

from sklearn.preprocessing import MinMaxScaler



2026-05-12 17:47:30.294705: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 17:47:30.304950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778608050.316609      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778608050.319976      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 17:47:30.332953: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
EPOCHS = 5
NUM_IMAGES = 150
BATCH_SIZE = 4
FOLDS = 5
IMAGE_DIM = (NUM_IMAGES, 55, 55)

COMP_DIR = "../input/osic-pulmonary-fibrosis-progression/"
TRAIN_PATH = "../input/osic-pulmonary-fibrosis-progression/train"
TEST_PATH = "../input/osic-pulmonary-fibrosis-progression/test"
SUB_PATH = "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"



In [3]:
comp_dir = "../input/osic-pulmonary-fibrosis-progression"

train_data = pd.read_csv(os.path.join(comp_dir, "train.csv"))
sub_raw = pd.read_csv(os.path.join(comp_dir, "sample_submission.csv"))
test_data = pd.read_csv(os.path.join(comp_dir, "test.csv"))

train_data = train_data.drop_duplicates(
    keep="first", subset=["Patient", "Weeks"]
).reset_index(drop=True)



In [4]:
train_data_u = train_data.drop_duplicates(subset=["Patient"]).copy()
train_data_u = train_data_u.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
train_data_u["Typical_FVC"] = (
    train_data_u["Base_FVC"].values / train_data_u["Base_Percent"].values
) * 100.0

train_data = train_data.merge(
    train_data_u.drop(["Age", "Sex", "SmokingStatus"], axis=1), on="Patient", how="left"
)



In [5]:
sub = sub_raw.copy()
sub["Patient"] = sub["Patient_Week"].apply(lambda x: x.split("_")[0])
sub["Weeks"] = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))

sub = sub[["Patient", "Weeks", "Patient_Week"]].copy()



In [6]:
test_data = test_data.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
test_data["Typical_FVC"] = (
    test_data["Base_FVC"].values / test_data["Base_Percent"].values
) * 100.0

sub = sub.merge(test_data, how="left", on="Patient")



In [7]:
train_data["Type"] = "train"
sub["Type"] = "test"

data = pd.concat([train_data, sub], axis=0, ignore_index=True)



In [8]:
prediction_col = ["FVC"]
Continuos_cols = [
    "Weeks",
    "Base_Week",
    "Base_FVC",
    "Typical_FVC",
    "Age",
    "Percent",
    "Base_Percent",
]

Categorical_cols = ["Sex", "SmokingStatus"]



In [9]:
# Bugfix: "FVC" is the target and not in Continuos_cols, so the previous attempt
# to pull scaling params for FVC from feature scaler raised KeyError.
# We keep feature scaling the same, but scale the target with a dedicated scaler
# so we can reliably invert predictions back to ml.
scaler_x = MinMaxScaler()
data[Continuos_cols] = scaler_x.fit_transform(data[Continuos_cols])

scaler_y = MinMaxScaler()
y_all = data[["FVC"]].astype(np.float32).values
scaler_y.fit(y_all)
fvc_min = float(scaler_y.data_min_[0])
fvc_max = float(scaler_y.data_max_[0])
fvc_scale = fvc_max - fvc_min if (fvc_max - fvc_min) != 0.0 else 1.0



In [10]:
sex_m = np.zeros((len(data), 1), dtype=np.float32)
sex_f = np.zeros((len(data), 1), dtype=np.float32)
sm_es = np.zeros((len(data), 1), dtype=np.float32)
sm_ns = np.zeros((len(data), 1), dtype=np.float32)
sm_cs = np.zeros((len(data), 1), dtype=np.float32)

sex_vals = data["Sex"].astype(str).values
smoke_vals = data["SmokingStatus"].astype(str).values

for i in range(len(data)):
    if sex_vals[i] == "Male":
        sex_m[i, 0] = 1.0
    elif sex_vals[i] == "Female":
        sex_f[i, 0] = 1.0

for i in range(len(data)):
    if smoke_vals[i] == "Ex-smoker":
        sm_es[i, 0] = 1.0
    elif smoke_vals[i] == "Never smoked":
        sm_ns[i, 0] = 1.0
    else:
        sm_cs[i, 0] = 1.0

data["sex_m"] = sex_m.reshape(-1)
data["sex_f"] = sex_f.reshape(-1)
data["sm_es"] = sm_es.reshape(-1)
data["sm_ns"] = sm_ns.reshape(-1)
data["sm_cs"] = sm_cs.reshape(-1)



In [11]:
x_cols = [
    "Weeks",
    "Base_Week",
    "Base_FVC",
    "Age",
    "sex_m",
    "sex_f",
    "sm_es",
    "sm_ns",
    "sm_cs",
]

x_train = data.loc[data["Type"] == "train", x_cols].values.astype(np.float32)

# Scale y using scaler_y for consistency with inverse-transform used at inference.
y_train_fvc = scaler_y.transform(
    data.loc[data["Type"] == "train", prediction_col].values.astype(np.float32)
).astype(np.float32)

x_test = data.loc[data["Type"] == "test", x_cols].values.astype(np.float32)

y_train = np.repeat(y_train_fvc, 3, axis=1).astype(np.float32)



In [12]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.mean(metric)


def qloss(y_true, y_pred):
    qs = [0.2, 0.50, 0.8]
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q * e, (q - 1) * e)
    return K.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1.0 - _lambda) * score(y_true, y_pred)

    return loss




2026-05-12 17:47:36.401397: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [13]:
def build_dense_model(n_features: int):
    inp = L.Input(shape=(n_features,), name="tabular_input")
    x = L.Dense(128, activation="relu")(inp)
    x = L.Dense(64, activation="relu")(x)
    x = L.Dense(32, activation="relu")(x)
    out = L.Dense(3, activation="linear", name="quantiles")(x)
    model = M.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=mloss(0.8))
    return model


tf.keras.utils.set_random_seed(42)

model = build_dense_model(x_train.shape[1])
model.fit(x_train, y_train, batch_size=32, epochs=EPOCHS, verbose=1)



Epoch 1/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.0128   
Epoch 2/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - loss: 0.9507
Epoch 3/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 845us/step - loss: 0.9380
Epoch 4/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 856us/step - loss: 0.9329
Epoch 5/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 859us/step - loss: 0.9316


In [14]:
pred = model.predict(x_test, verbose=1).astype(np.float32)

# Model outputs are in scaled-FVC space -> invert to ml for submission
fvc_pred_scaled = pred[:, 1]
fvc_pred = fvc_pred_scaled * fvc_scale + fvc_min

# Confidence should be in ml as well; scale the predicted spread accordingly.
conf_scaled = (pred[:, 2] - pred[:, 0]).astype(np.float32)
conf = conf_scaled * fvc_scale
conf = np.maximum(conf, 70.0)



60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 810us/step


In [15]:
sub_out = sub.copy()
sub_out["FVC"] = fvc_pred
sub_out["Confidence"] = conf

subm = sub_out[["Patient_Week", "FVC", "Confidence"]].copy()
subm["FVC"] = subm["FVC"].astype(np.float32)
subm["Confidence"] = subm["Confidence"].astype(np.float32)

subm.to_csv("submission.csv", index=False)



In [16]:
subm.head()

,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2472.615479,202.061661
1,ID00126637202218610655908_-2,2466.728271,203.636536
2,ID00126637202218610655908_-1,2460.840820,205.211533
3,ID00126637202218610655908_0,2454.953369,206.787048
4,ID00126637202218610655908_1,2449.065918,208.362045
